# NeMo-Skills Competitive Programmer Finetuning

This notebook prepares the OpenCodeReasoning-2 dataset (CPP split) and finetunes a Qwen2.5-Coder-7B-Instruct model. Storage management is optimized to handle large base datasets.

In [1]:
%pip install git+https://github.com/NVIDIA-NeMo/Skills.git
%pip install datasets==2.16.0

  Cloning https://github.com/NVIDIA-NeMo/Skills.git to /tmp/pip-req-build-hr9ffqfd
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA-NeMo/Skills.git /tmp/pip-req-build-hr9ffqfd
  Resolved https://github.com/NVIDIA-NeMo/Skills.git to commit acec1b0c6e7614c5a35095e8e192be89b0fa02f7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/NVIDIA/compute-eval.git (to revision e01a5d2) to /tmp/pip-install-k9fnxrs7/compute-eval_60ea6d2acc944bff9ab759773a6bef90
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA/compute-eval.git /tmp/pip-install-k9fnxrs7/compute-eval_60ea6d2acc944bff9ab759773a6bef90
  Running command git checkout -q e01a5d2
  Resolved https://github.com/NVIDIA/compute-eval.git to commit e01a5d2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 9.0 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:
      Successfully uninstalled multiprocess-0.70.16
  Attempting uninstall: datasets
 

In [ ]:
!ns setup
# /content:/workspace, /content/hf_cache:/hf_models
# /hf_models
# PYTHONPATH=/nemo_run/code

In [ ]:
import os
from huggingface_hub import snapshot_download
os.environ["NEMO_SKILLS_CONFIGS"] = "/content/cluster_configs"
os.environ["PYTHONPATH"] = "/content/Skills"
os.environ["NVIDIA_API_KEY"] = "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"
os.environ["HF_TOKEN"] = "xxx"  # huggingface.co/settings/tokens
MY_TOKEN = "xxx"

In [11]:
%cd /content/Skills
!git pull

/content/Skills
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 1.55 KiB | 529.00 KiB/s, done.
From https://github.com/Belaleatsbanana/Skills
   053f0073..4fecd6fb  main       -> origin/main
Updating 053f0073..4fecd6fb
Fast-forward
 nemo_notebook.ipynb                                | 111 ++++++++++++++++++---
 .../opencodereasoning/scripts/prepare_questions.py |   8 +-
 2 files changed, 100 insertions(+), 19 deletions(-)


In [3]:
import datasets
print(datasets.__version__)

4.8.4


In [3]:
%cd /content
!git clone https://github.com/Belaleatsbanana/Skills.git
%cd /content/Skills

/content
Cloning into 'Skills'...
remote: Enumerating objects: 40778, done.
remote: Counting objects: 100% (1887/1887), done.
remote: Compressing objects: 100% (860/860), done.
remote: Total 40778 (delta 1549), reused 1070 (delta 1024), pack-reused 38891 (from 4)
Receiving objects: 100% (40778/40778), 383.35 MiB | 53.15 MiB/s, done.
Resolving deltas: 100% (29538/29538), done.
/content/Skills


In [3]:
ls

cluster_configs/  hf_cache/  opencodereasoning/  sample_data/  Skills/


## Step 1: Data Preparation

Extract questions from the OCR2 dataset. Original solutions are intentionally skipped as we will generate higher quality reasoning traces using a strong model (DeepSeek-R1).

In [12]:
!HF_HUB_ENABLE_HF_TRANSFER=1 python /content/Skills/recipes/opencodereasoning/scripts/prepare_questions.py \
    --output_dir /content/opencodereasoning/data \
    --cache_dir /content/hf_cache

[STEP 1] Loading OpenCodeReasoning-2 'cpp' split...
Traceback (most recent call last):
  File "/content/Skills/recipes/opencodereasoning/scripts/prepare_questions.py", line 99, in <module>
    ocr2_dataset = load_dataset(
                   ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1708, in load_dataset
    builder_instance.download_and_prepare(
  File "/usr/local/lib/python3.12/dist-packages/datasets/builder.py", line 884, in download_and_prepare
    self._download_and_prepare(
  File "/usr/local/lib/python3.12/dist-packages/datasets/builder.py", line 925, in _download_and_prepare
    split_generators: list[SplitGenerator] = self._split_generators(dl_manager, **split_generators_kwargs)
                                             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/packaged_modules/parquet/parquet.py", line 129, in _split_generators
    raise ValueError(
Value

## Step 2: Generate Solutions

Generate synthetic solutions with DeepSeek-R1 via NVIDIA NIM to get high-quality reasoning traces.

In [8]:
# This uses the 'demo' mode which connects to NVIDIA NIM (requires NVIDIA_API_KEY environment variable)
!python /content/Skills/recipes/opencodereasoning/pipeline/prepare_solutions.py --mode demo

Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 45, in <module>
    PACKAGE_DISTRIBUTION_MAPPING = importlib.metadata.packages_distributions()
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 947, in packages_distributions
    for pkg in _top_level_declared(dist) or _top_level_inferred(dist):
                                            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/metadata/__init__.py", line 959, in _top_level_inferred
    for f in always_iterable(dist.files)
                             

## Step 3: Prepare SFT Data

Apply formatting for finetuning using the filtered R1 outputs.

In [ ]:
# Pointing to the filtered output from the R1 generation pipeline
input_path = "/workspace/opencodereasoning/data/solution-sdg-toy/filtered/output.jsonl"
!python -m nemo_skills.training.prepare_data \
    ++input_files={input_path} \
    ++output_path=/workspace/opencodereasoning/sft-data.jsonl \
    ++prompt_config=generic/math \
    ++add_unlabeled=true

## Step 4: Finetune Qwen2.5-Coder-7B-Instruct

Train using NeMo-RL.

In [ ]:
!ns nemo_rl sft \
    --cluster=slurm \
    --training_data=/workspace/opencodereasoning/sft-data.jsonl \
    --hf_model=Qwen/Qwen2.5-Coder-7B-Instruct \
    --backend=megatron \
    --expname=qwen-coder-sft \
    ++policy.max_total_sequence_length=8192 \
    ++sft.max_num_epochs=2